# Train the prostate lesion model

This notebook clones the source code into Colab's local storage, extracts the separately archived processed splits locally for fast loading, and saves model checkpoints to Google Drive.

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from urllib.parse import urlparse, unquote
from google.colab import output
from google.colab import drive


automatic branch detection

In [ ]:
drive.mount('/content/drive')

project_dir = Path('/content/Masters_quad_joined_prostate_detection')
repository_url = 'https://github.com/Peter-worm/Masters_quad_joined_prostate_detection.git'

# Define the branch you want to clone here
branch_name = 'feat/baseline-training' 

if not (project_dir / '.git').is_dir():
    if project_dir.exists():
        raise FileExistsError(
            f'{project_dir} exists but is not a Git checkout.'
        )
    clone_result = subprocess.run(
        ['git', 'clone', '-b', branch_name, '--depth', '1', repository_url, str(project_dir)],
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            f'Unable to clone {repository_url} (branch: {branch_name}):\n{clone_result.stderr.strip()}'
        )

os.chdir(project_dir)
print(f'Project directory: {project_dir}')
print(f'Active branch: {branch_name}')

## Optional: update the local checkout

Run the next cell after pushing code changes to GitHub and before importing project modules. Set `UPDATE_REPOSITORY` to `False` when you want to keep the current checkout.

In [ ]:
UPDATE_REPOSITORY = True

if UPDATE_REPOSITORY and (project_dir / '.git').is_dir():
    update_result = subprocess.run(
        ['git', '-C', str(project_dir), 'pull', '--ff-only'],
        capture_output=True,
        text=True,
    )
    if update_result.returncode != 0:
        raise RuntimeError(
            'Unable to update the local checkout. Commit or discard local '
            f'changes, then retry:\n{update_result.stderr.strip()}'
        )
    print(update_result.stdout.strip() or 'Repository is already up to date.')


In [ ]:
%pip install -r requirements.txt


In [ ]:
drive_archive_dir = Path('/content/drive/MyDrive/Master_Thesis/processed_archives')
local_processed_dir = project_dir / 'data' / 'processed'
weights_dir = Path('/content/drive/MyDrive/Master_Thesis/training_checkpoints')

if local_processed_dir.exists():
    shutil.rmtree(local_processed_dir)
local_processed_dir.mkdir(parents=True, exist_ok=True)

for split in ('train', 'test'):
    archive_path = drive_archive_dir / f'processed_{split}.tar.gz'
    if not archive_path.is_file():
        raise FileNotFoundError(
            f'Missing processed archive: {archive_path}. '
            'Run 01_preprocessing.ipynb first.'
        )
    split_dir = local_processed_dir / split
    split_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as archive:
        archive.extractall(split_dir)
    extracted_files = list(split_dir.glob('*.pkl'))
    if not extracted_files:
        raise RuntimeError(f'Archive contains no pickle files: {archive_path}')
    print(f'Extracted {len(extracted_files)} files to {split_dir}')
weights_dir.mkdir(parents=True, exist_ok=True)
os.environ['AI_PROJECT_DIR'] = str(project_dir)
os.environ['SAVED_WEIGHTS_DIR'] = str(weights_dir)
sys.path.insert(0, str(project_dir))


## Start training

The current training implementation reports the `test` split as validation during training. Treat the final test score accordingly.

In [ ]:
from src.models.train import train

train()
